# 03 - Module 3 prep: gold-layer views for the dashboard

Hands-on companion for [`module-3-dashboard-and-wrapup.md`](../module-3-dashboard-and-wrapup.md).

Most of Module 3 happens in the Databricks UI (AI/BI Dashboard, Genie
space, Catalog Explorer). This notebook only does the SQL piece:
create the three **gold-layer views** the dashboard binds to, plus a
few governance sanity checks.

**SSIS parallel.** Each view replaces a stored procedure that an SSRS
report would have called. Same idea (one source of truth per report
region), governed by Unity Catalog instead of SQL Server permissions.


In [ ]:
%sql
USE workspace.bi_course;


## 1. `v_kpi_revenue_by_product`


In [ ]:
%sql
CREATE OR REPLACE VIEW v_kpi_revenue_by_product AS
SELECT
    p.product_name,
    SUM(f.revenue)  AS total_revenue,
    SUM(f.quantity) AS units_sold
FROM fact_sales  f
JOIN dim_product p USING (product_id)
GROUP BY p.product_name;


## 2. `v_kpi_daily_revenue`


In [ ]:
%sql
CREATE OR REPLACE VIEW v_kpi_daily_revenue AS
SELECT order_date, SUM(revenue) AS daily_revenue
FROM   fact_sales
GROUP BY order_date;


## 3. `v_kpi_revenue_by_region_product`


In [ ]:
%sql
CREATE OR REPLACE VIEW v_kpi_revenue_by_region_product AS
SELECT
    r.region_name,
    p.product_name,
    SUM(f.revenue) AS revenue
FROM fact_sales  f
JOIN dim_region  r USING (region_id)
JOIN dim_product p USING (product_id)
GROUP BY r.region_name, p.product_name;


## 4. Sanity-check the three views

Each `SELECT` should return rows with no errors. Use these as the
starting queries when you build the corresponding dashboard tile.


In [ ]:
%sql
SELECT * FROM v_kpi_revenue_by_product        ORDER BY total_revenue DESC LIMIT 10;


In [ ]:
%sql
SELECT * FROM v_kpi_daily_revenue             ORDER BY order_date;


In [ ]:
%sql
SELECT * FROM v_kpi_revenue_by_region_product ORDER BY region_name, revenue DESC;


## 5. Governance walkthrough (Unity Catalog)


In [ ]:
%sql
SHOW GRANTS ON SCHEMA workspace.bi_course;


In [ ]:
%sql
SHOW GRANTS ON TABLE  workspace.bi_course.fact_sales;


On Free Edition you are the only principal, so `SHOW GRANTS` will list
yourself as owner. Open **Catalog Explorer** in the left rail and look
at the **Lineage** tab for `v_kpi_revenue_by_product` - you should see
`fact_sales` and `dim_product` upstream, and (after you build it) your
dashboard downstream.


## You're done

Switch to the **Dashboards** section in the left rail and follow
[`module-3-dashboard-and-wrapup.md`](../module-3-dashboard-and-wrapup.md)
to build the AI/BI Dashboard and the AI/BI Genie space on top of these views.
